# Small Model Supremacy — Full Training Pipeline

Fine-tuning Qwen2.5-1.5B with QLoRA to outperform GPT-4o on structured data extraction.

**Requirements:** Kaggle GPU T4 x2 enabled

**Expected runtime:** ~3-4 hours for full training

## 1. Setup

In [ ]:
!git clone https://github.com/BhavanaR006/small-model-supremacy.git
%cd small-model-supremacy
!pip install trl peft bitsandbytes accelerate transformers datasets --quiet

## 2. Load Model + Apply QLoRA

In [ ]:
import json, torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Load model with 4-bit NF4 quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B", trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

# Apply LoRA adapters
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=32, lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 3. Prepare Dataset

In [ ]:
# Load ALL training data (1500 examples)
train_data = [json.loads(l) for l in open("data/train.jsonl")]
val_data = [json.loads(l) for l in open("data/val.jsonl")]

# Format with strict JSON instruction using Qwen chat template
def format_strict(ex):
    output_json = json.dumps(ex['expected_output'], ensure_ascii=False)
    text = f"""<|im_start|>system
You are a JSON extraction assistant. You ONLY output valid JSON. No explanations.<|im_end|>
<|im_start|>user
Extract from the following text into the schema \"{ex['schema_id']}\".

Text: {ex['input_text']}<|im_end|>
<|im_start|>assistant
{output_json}<|im_end|>"""
    return {"text": text}

train_dataset = Dataset.from_list([format_strict(ex) for ex in train_data])
val_dataset = Dataset.from_list([format_strict(ex) for ex in val_data])

# Tokenize
def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")

train_tok = train_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
val_tok = val_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

def add_labels(examples):
    examples["labels"] = examples["input_ids"].copy()
    return examples

train_tok = train_tok.map(add_labels, batched=True)
val_tok = val_tok.map(add_labels, batched=True)
train_tok.set_format("torch")
val_tok.set_format("torch")

print(f"Dataset ready: {len(train_tok)} train, {len(val_tok)} val")

## 4. Train (Full — 1500 examples, 5 epochs)

In [ ]:
training_args = TrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=10,
    save_steps=100,
    eval_strategy="steps",
    eval_steps=100,
    save_total_limit=3,
    bf16=True,
    max_grad_norm=1.0,
    report_to="none",
    dataloader_num_workers=2,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

print("Starting full training (1500 examples, 5 epochs)...")
print("Estimated time: ~3-4 hours on T4. Progress bar will show below.")
trainer.train()
trainer.save_model("./checkpoints/final")
tokenizer.save_pretrained("./checkpoints/final")
print("\n✓ Training complete! Model saved to ./checkpoints/final")

## 5. Test the Fine-Tuned Model

In [ ]:
model.eval()

def extract(text, schema_id):
    """Extract structured JSON from text using the fine-tuned model."""
    prompt = f"""<|im_start|>system
You are a JSON extraction assistant. You ONLY output valid JSON. No explanations.<|im_end|>
<|im_start|>user
Extract from the following text into the schema \"{schema_id}\".

Text: {text}<|im_end|>
<|im_start|>assistant
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    stop_token_id = tokenizer.encode("<|im_end|>", add_special_tokens=False)[0]
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
            eos_token_id=stop_token_id,
        )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(generated, skip_special_tokens=True)
    
    # Extract first JSON object
    try:
        start = raw.index('{')
        depth = 0
        for i in range(start, len(raw)):
            if raw[i] == '{': depth += 1
            elif raw[i] == '}': depth -= 1
            if depth == 0:
                return json.loads(raw[start:i+1])
    except:
        return raw

# Test on examples the model hasn't seen
print("=" * 60)
print("MODEL EVALUATION — Unseen Examples")
print("=" * 60)

tests = [
    ("Dr. Sarah Chen presented her research on quantum error correction at NeurIPS 2025 in Vancouver, Canada.", "conference_talk_simple"),
    ("Professor James Miller discussed autonomous drone navigation at ICRA 2025 in Tokyo, Japan.", "conference_talk_simple"),
    ("The TechNova UltraFit Pro Wireless Earbuds feature Bluetooth 5.3. Priced at 79.99 USD, 15% off. New, in stock. 5x3x2cm, 0.05kg. SKU: TN-4521-A01.", "product_listing_medium"),
    ("At the World AI Summit 2025 in Dubai, Dr. Amina Hassan delivered a keynote on federated learning for healthcare applications.", "conference_talk_simple"),
]

for text, schema in tests:
    print(f"\nSchema: {schema}")
    print(f"Input: {text[:80]}...")
    result = extract(text, schema)
    print(f"Output: {json.dumps(result, indent=2)}")
    print()

## 6. Save Model for Download

In [ ]:
# Zip the final model for easy download
import shutil
shutil.make_archive('/kaggle/working/model_final', 'zip', './checkpoints/final')
print("Model archived to /kaggle/working/model_final.zip")
print("Download it from the Output tab on the right panel.")